# Learn VUS curve artifact: build and save

This notebook creates one reusable curve artifact from labels and anomaly scores.

The artifact must not depend on a selected FPR budget.

Type the code marked with `# TODO`, run the cell, and inspect the result before continuing.

## 1. Imports and paths

Use the repository `.venv` when launching Jupyter.

The private metric helpers are used here only to make the learning notebook follow the current implementation.

In [1]:
import sys
from pathlib import Path

# warning: do not add PosixPath objects into sys.path
# use str(...) to add strings instead.
sys.path.append(str(Path.cwd().parent.parent))
sys.path = list(set(sys.path))
sys.path

['',
 '/Users/conquerormikrokosmos/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python312.zip',
 '/Users/conquerormikrokosmos/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/lib-dynload',
 '/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/.venv/lib/python3.12/site-packages',
 '/Users/conquerormikrokosmos/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12',
 '/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026']

In [2]:
import json
import numpy as np

from src.metrics.pointwise import (
    FPR_BUDGETS,
    _build_score_thresholds,
    _compute_range_precision_recall,
    _compute_range_roc_rates,
)

REPO_ROOT = Path.cwd().parent.parent
assert (REPO_ROOT / "src").is_dir(), "Launch Jupyter from the repository root."
ARTIFACT_DIR = REPO_ROOT / "notebooks" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_NPZ = ARTIFACT_DIR / "vus_curve_learning_demo.npz"
ARTIFACT_JSON = ARTIFACT_DIR / "vus_curve_learning_demo.json"

## 2. Tiny deterministic input

Use a tiny example first so every array can be inspected manually.

In [3]:
point_labels = np.array([0, 0, 1, 1], dtype=np.int64)
point_scores = np.array([0.10, 0.20, 0.80, 0.90], dtype=np.float64)
max_buffer_size = 1
num_thresholds = 5

# TODO: print the two shapes and verify that labels and scores have equal length.
print(point_labels.shape)
print(point_scores.shape)
print(
    "labels and scores have equal length?",
    point_labels.shape[0] == point_scores.shape[0],
)

assert point_labels.ndim == 1
assert point_scores.ndim == 1
assert point_labels.shape == point_scores.shape
assert set(np.unique(point_labels)).issubset({0, 1})
assert np.all(np.isfinite(point_scores))
assert max_buffer_size >= 0
assert num_thresholds > 0

(4,)
(4,)
labels and scores have equal length? True


## 3. Build the shared threshold axis

This threshold axis will be reused for PR and ROC values.

In [4]:
# TODO: call _build_score_thresholds with point_scores and num_thresholds.
thresholds = _build_score_thresholds(point_scores, num_thresholds)

# TODO: print thresholds, its shape, and its dtype.
print(thresholds)
print(thresholds.shape)
print(type(thresholds))

assert thresholds is not ...
thresholds = np.asarray(thresholds, dtype=np.float64)
assert thresholds.ndim == 1
assert thresholds.size >= 2
assert np.all(np.isfinite(thresholds))
assert thresholds[0] < np.min(point_scores)
assert thresholds[-1] > np.max(point_scores)
assert np.all(np.diff(thresholds) >= 0)

score_array w/ reshape(-1): [0.1 0.2 0.8 0.9]
score_array w/o reshape(-1): [0.1 0.2 0.8 0.9]
[0.1 0.1 0.2 0.8 0.9 0.9]
(6,)
<class 'numpy.ndarray'>


## 4. Inspect one buffer size

First compute one PR and ROC point for one threshold.

Do not write a threshold loop yet.

In [7]:
buffer_size = 0
threshold = thresholds[3]  # TODO: choose one value from thresholds.

# TODO: call _compute_range_precision_recall.
precision, recall = _compute_range_precision_recall(
    point_labels, point_scores, threshold, buffer_size
)

# TODO: call _compute_range_roc_rates.
false_positive_rate, true_positive_rate = _compute_range_roc_rates(
    point_labels, point_scores, threshold, buffer_size
)

# TODO: print all four values.
print("precision=", precision)
print("recall=", recall)
print("false_positive_rate=", false_positive_rate)
print("true_positive_rate=", true_positive_rate)

assert threshold is not ...
assert np.isfinite(threshold)
assert np.isfinite(precision) and 0.0 <= precision <= 1.0
assert np.isnan(recall) or 0.0 <= recall <= 1.0
assert np.isnan(false_positive_rate) or 0.0 <= false_positive_rate <= 1.0
assert np.isnan(true_positive_rate) or 0.0 <= true_positive_rate <= 1.0

precision= 1.0
recall= 0.5
false_positive_rate= 0.0
true_positive_rate= 0.5


## 5. Build one-buffer curve points

The four lists must have exactly the same length as `thresholds`.

In [ ]:
precision_values = []
recall_values = []
false_positive_rates = []
true_positive_rates = []

# TODO: loop through thresholds.
# TODO: append one PR pair and one ROC pair for every threshold.

# TODO: assert that all four lists have equal length.
assert len(precision_values) == len(thresholds)
assert len(recall_values) == len(thresholds)
assert len(false_positive_rates) == len(thresholds)
assert len(true_positive_rates) == len(thresholds)
assert all(np.isnan(value) or 0.0 <= value <= 1.0 for value in precision_values)
assert all(np.isnan(value) or 0.0 <= value <= 1.0 for value in recall_values)
assert all(np.isnan(value) or 0.0 <= value <= 1.0 for value in false_positive_rates)
assert all(np.isnan(value) or 0.0 <= value <= 1.0 for value in true_positive_rates)

## 6. Build the complete in-memory artifact

Use one row per buffer size and one column per threshold.

The same artifact will later serve full VUS and all budgeted VUS reducers.

In [ ]:
buffer_sizes = np.arange(max_buffer_size + 1, dtype=np.int64)

precision_matrix = []
recall_matrix = []
false_positive_rate_matrix = []
true_positive_rate_matrix = []

# TODO: loop through buffer_sizes.
# TODO: inside that loop, create four fresh lists.
# TODO: loop through thresholds and append the four range metrics.
# TODO: append the four completed lists to the four matrices.

# TODO: convert the four matrices to NumPy arrays with dtype float64.
precision_matrix = ...
recall_matrix = ...
false_positive_rate_matrix = ...
true_positive_rate_matrix = ...

# TODO: assert that each matrix has shape (max_buffer_size + 1, len(thresholds)).
expected_shape = (max_buffer_size + 1, thresholds.size)
assert buffer_sizes.shape == (max_buffer_size + 1,)
assert precision_matrix.shape == expected_shape
assert recall_matrix.shape == expected_shape
assert false_positive_rate_matrix.shape == expected_shape
assert true_positive_rate_matrix.shape == expected_shape
assert all(
    array.dtype == np.float64
    for array in (
        precision_matrix,
        recall_matrix,
        false_positive_rate_matrix,
        true_positive_rate_matrix,
    )
)
assert np.all(
    np.isnan(precision_matrix) | ((0.0 <= precision_matrix) & (precision_matrix <= 1.0))
)
assert np.all(
    np.isnan(recall_matrix) | ((0.0 <= recall_matrix) & (recall_matrix <= 1.0))
)
assert np.all(
    np.isnan(false_positive_rate_matrix)
    | ((0.0 <= false_positive_rate_matrix) & (false_positive_rate_matrix <= 1.0))
)
assert np.all(
    np.isnan(true_positive_rate_matrix)
    | ((0.0 <= true_positive_rate_matrix) & (true_positive_rate_matrix <= 1.0))
)

## 7. Save numeric arrays and metadata

Use NPZ for numeric arrays and JSON for human-readable metadata.

The budget values belong in metadata, but they must not change the curve arrays.

In [ ]:
# TODO: save buffer_sizes, thresholds, and the four matrices with np.savez_compressed.

metadata = {
    "artifact_version": "v1-learning",
    "num_thresholds": ...,  # TODO
    "max_buffer_size": ...,  # TODO
    "fpr_budgets": ...,  # TODO: use list(FPR_BUDGETS)
}

# TODO: write metadata as JSON text.
assert metadata["artifact_version"]
assert metadata["num_thresholds"] == num_thresholds
assert metadata["max_buffer_size"] == max_buffer_size
assert tuple(metadata["fpr_budgets"]) == tuple(FPR_BUDGETS)

print(ARTIFACT_NPZ)
print(ARTIFACT_JSON)

## 8. Reload and verify

Do not continue until the saved artifact reloads with the expected keys and shapes.

In [ ]:
# TODO: load the NPZ artifact.
loaded_arrays = ...

# TODO: load the JSON metadata.
loaded_metadata = ...

# TODO: print keys, metadata, and matrix shapes.
# TODO: assert that loaded values equal the values saved above.
assert set(loaded_arrays.files) == {
    "buffer_sizes",
    "thresholds",
    "precision_matrix",
    "recall_matrix",
    "false_positive_rate_matrix",
    "true_positive_rate_matrix",
}
assert loaded_metadata["artifact_version"] == metadata["artifact_version"]
assert loaded_metadata["num_thresholds"] == metadata["num_thresholds"]
assert loaded_metadata["max_buffer_size"] == metadata["max_buffer_size"]
assert loaded_metadata["fpr_budgets"] == list(metadata["fpr_budgets"])
assert np.array_equal(loaded_arrays["buffer_sizes"], buffer_sizes)
assert np.array_equal(loaded_arrays["thresholds"], thresholds)
assert np.allclose(loaded_arrays["precision_matrix"], precision_matrix, equal_nan=True)
assert np.allclose(loaded_arrays["recall_matrix"], recall_matrix, equal_nan=True)
assert np.allclose(
    loaded_arrays["false_positive_rate_matrix"],
    false_positive_rate_matrix,
    equal_nan=True,
)
assert np.allclose(
    loaded_arrays["true_positive_rate_matrix"],
    true_positive_rate_matrix,
    equal_nan=True,
)